# Lesson 4a: Regularisation and Generalisation — Theory

3a/3b made sure a deep network trains reliably. This notebook asks the next
question: once it trains, does it generalise — does low training error
imply low *test* error? Classical statistics has a clean answer, the
bias-variance tradeoff, and modern deep networks routinely violate it: they
are trained with vastly more parameters than training examples, sit at (or
past) zero training error, and often generalise well anyway. This notebook
derives the classical picture, four concrete techniques that constrain a
network's effective capacity or inject useful noise (weight decay,
dropout, early stopping, data augmentation), and the phenomenon that
extends the classical picture into the overparameterised regime: double
descent.

By the end of this notebook you will have:
- derived the classical **bias-variance decomposition** and identified
  exactly where it stops predicting what overparameterised networks
  actually do,
- proven that **weight decay is algebraically identical to L2
  regularisation** under plain SGD, and shown precisely where that
  identity breaks for adaptive optimisers,
- derived **dropout** as training an implicit, exponentially large,
  weight-sharing ensemble, and stated the **train/test scaling rule** that
  makes a single forward pass at test time approximate that ensemble's
  average,
- used **early stopping** on a single training run's own train/test curve,
  no extra training required,
- implemented **data augmentation** (random crop and flip) from scratch and
  measured its effect on the same overfitting network, and
- reproduced a **double descent** test-error curve on real CIFAR-10 data,
  showing test error rise then fall again as model capacity crosses the
  interpolation threshold.

## Introduction

The **generalisation gap** — test error minus training error — is the
central quantity of this notebook, the same way the per-layer gradient
norm was the central diagnostic of 3a. Classical statistical learning
theory frames it through **bias and variance**: a model with too little
capacity cannot fit the true relationship even with unlimited data (high
bias, underfitting); a model with too much capacity fits the training
noise as well as the signal (high variance, overfitting); total expected
error is minimised at some intermediate capacity, producing the textbook
U-shaped test-error-vs-capacity curve.

Every technique in this notebook — weight decay, dropout, early stopping,
data augmentation — is a different way of pulling a network back from the
high-variance side of that curve without literally removing parameters:
each constrains the *effective* capacity the optimiser is free to use, or
injects noise that makes memorising the training set less useful than
learning the underlying pattern. The notebook closes by confronting the
place the classical U-shape stops predicting reality: modern networks are
trained with orders of magnitude more parameters than training examples,
routinely reach zero training error, and often still generalise
well — the "double descent" phenomenon precisely characterises what the
classical picture misses.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, dropout masks, augmentation, data subsampling) is reproducible.
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# CIFAR-10 via the Hugging Face parquet mirror (see 3a: the canonical
# torchvision host measured unreliably slow in this environment).
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float64) / 255.0
        for i in idx
    ])  # (n, 32, 32, 3), pixel values in [0, 1]
    labels = df.iloc[idx]["label"].to_numpy().astype(int)
    return images, labels


def one_hot(labels, n_classes=10):
    Y = np.zeros((n_classes, len(labels)))
    Y[labels, np.arange(len(labels))] = 1.0
    return Y


CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

# Main working set: deliberately small (300 train) relative to the network's
# capacity below, so it overfits within a modest number of epochs -- every
# regularisation technique in this notebook is demonstrated against this one
# overfitting baseline.
N_TRAIN, N_TEST = 300, 300
images_train, labels_train = load_cifar10_subset("train", N_TRAIN, seed=SEED)
images_test, labels_test = load_cifar10_subset("test", N_TEST, seed=SEED + 1)
X_train = images_train.reshape(N_TRAIN, -1).T
X_test = images_test.reshape(N_TEST, -1).T
Y_train, Y_test = one_hot(labels_train), one_hot(labels_test)

# A second, much smaller set purely for the "Bias and Variance"/"Double
# Descent" random-features experiment, where the interpolation threshold
# needs to be reachable by a model capacity we can actually sweep past.
N_DD_TRAIN = 50
images_dd, labels_dd = load_cifar10_subset("train", N_DD_TRAIN, seed=SEED + 2)
X_dd_train = images_dd.reshape(N_DD_TRAIN, -1).T
Y_dd_train = one_hot(labels_dd)

print("X_train:", X_train.shape, " X_test:", X_test.shape, " X_dd_train:", X_dd_train.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(images_train[i])
    ax.set_title(CLASS_NAMES[labels_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.show()

## Bias and Variance in Deep Networks

For a regression target $y = f(x) + \varepsilon$ with noise
$\varepsilon$, zero mean, variance $\sigma^2$, and an estimator
$\hat f_D(x)$ trained on a random training set $D$, decompose the expected
squared error at $x$ by inserting and subtracting $\bar f(x) :=
\mathbb E_D[\hat f_D(x)]$:

$$\mathbb E_{D,\varepsilon}\!\left[(y-\hat f_D(x))^2\right]
= \underbrace{\left(f(x)-\bar f(x)\right)^2}_{\textbf{Bias}^2}
+ \underbrace{\mathbb E_D\!\left[(\hat f_D(x)-\bar f(x))^2\right]}_{\textbf{Variance}}
+ \underbrace{\sigma^2}_{\text{irreducible noise}}.$$

(The cross term vanishes because $\mathbb E_D[\hat f_D(x)-\bar f(x)]=0$ by
definition of $\bar f$, and $\varepsilon$ is independent of $\hat f_D$.)
**Bias** measures how far the *average* model (over all possible training
sets) is from the truth — high when the model class is too simple to
represent $f$. **Variance** measures how much the model *changes* from one
training set to another — high when the model is flexible enough to fit
whatever noise happens to be in that particular training set. Increasing
model capacity (more parameters, deeper networks) typically decreases bias
and increases variance, producing the textbook U-shaped total-error curve
against capacity, with a sweet spot in between.

**Where this classical picture breaks for deep networks:** the argument
implicitly assumes variance keeps growing as capacity grows without bound.
Real deep networks are trained with capacity far past the point where
training error reaches (near) zero — the **interpolation threshold**,
where the number of parameters roughly matches the number of training
examples — and classical theory predicts variance, and hence test error,
should be at its worst there or beyond. Empirically it is not: pushing
capacity well past the interpolation threshold, test error frequently
comes back *down* again. The cell below runs a small experiment — a random
non-linear feature map fit to real CIFAR-10 images by minimum-norm least
squares, sweeping the number of features from far below to far above the
number of training examples — and, for now, shows only the classical
region: capacity approaching the interpolation threshold from below.
"Double Descent" revisits this exact experiment and continues the sweep
past it.

In [ ]:
def random_relu_features(X, W, b):
    return np.maximum(0.0, W @ X + b)  # (p, n)


def fit_min_norm(Phi_train, Y_train):
    # Y = Theta @ Phi has infinitely many exact solutions once p > n;
    # pinv picks the minimum-norm one -- the "ridgeless" regression estimator.
    return Y_train @ np.linalg.pinv(Phi_train)


def double_descent_point(p, seed):
    d = X_dd_train.shape[0]
    rng = np.random.default_rng(seed + p)
    W = rng.normal(0.0, 1.0 / np.sqrt(d), size=(p, d))
    b = rng.normal(0.0, 1.0, size=(p, 1))
    Phi_train = random_relu_features(X_dd_train, W, b)
    Phi_test = random_relu_features(X_test, W, b)
    Theta = fit_min_norm(Phi_train, Y_dd_train)
    train_err = ((Theta @ Phi_train).argmax(axis=0) != labels_dd).mean()
    test_err = ((Theta @ Phi_test).argmax(axis=0) != labels_test).mean()
    return train_err, test_err


dd_results = {}
for p in [5, 10, 20, 30, 40, 49]:
    dd_results[p] = double_descent_point(p, seed=SEED)

ps = sorted(dd_results)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(ps, [dd_results[p][0] for p in ps], marker="o", label="train error")
ax.plot(ps, [dd_results[p][1] for p in ps], marker="o", label="test error")
ax.axvline(N_DD_TRAIN, color="gray", linestyle=":", label=f"n = {N_DD_TRAIN} (interpolation threshold)")
ax.set_xlabel("number of random features (model capacity)")
ax.set_ylabel("classification error")
ax.set_title("Classical regime: capacity approaching n")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

Train error falls toward zero as capacity approaches $n=50$, exactly as
expected — more features fit the training set ever more exactly. Test
error, meanwhile, starts high (too few features to capture real structure:
high bias), improves in the middle, then rises again as capacity nears
$n$ — the classical U-shape, right on schedule. What happens if the sweep
keeps going past $p=n$ is the subject of "Double Descent" below.

## Weight Decay and L2

Adding an $L_2$ penalty $\tfrac{\lambda}{2}\lVert w\rVert^2$ to the loss
is the classical way to constrain capacity directly: it discourages large
weights, which (for a network with bounded activations) discourages the
sharp, high-variance functions that large weights can represent. Its
gradient contribution is $\lambda w$, so plain SGD on the penalised loss
gives

$$w \leftarrow w - \eta(\nabla_w J + \lambda w) = (1-\eta\lambda)\,w - \eta\nabla_w J.$$

The right-hand form — shrink $w$ by a constant factor $(1-\eta\lambda)$,
*then* take the ordinary gradient step — is exactly **weight decay**. For
plain SGD these are not merely similar, they are the same update, to
floating-point precision, verified below.

**The equivalence fails for adaptive optimisers.** Adam divides the update
by $\sqrt{\hat v}+\epsilon$, a per-parameter factor built from the running
average of *squared* gradients. If the $L_2$ term $\lambda w$ is folded
into the gradient before that division (the naive port of "add $L_2$ to
the loss" to Adam), it gets scaled by whatever $1/\sqrt{\hat v}$ happens to
be for that parameter — parameters with a history of large gradients have
their weight decay suppressed, parameters with small gradients have theirs
amplified, entirely accidentally. **Decoupled weight decay** (AdamW,
Loshchilov & Hutter, 2019) fixes this by keeping $\lambda w$ *out* of the
moment estimates entirely and subtracting $\eta\lambda w$ directly, so
every parameter gets the same proportional shrinkage regardless of its
gradient history — this is why "Adam with L2 regularisation" and "AdamW"
are different optimisers, not a naming quirk.

In [ ]:
rng = np.random.default_rng(SEED)
w0 = rng.normal(size=8)
grad = rng.normal(size=8)
lr, lam = 0.1, 0.2

# --- plain SGD: the two forms are the same update, verified to machine precision ---
w_l2_sgd = w0 - lr * (grad + lam * w0)
w_decay_sgd = (1 - lr * lam) * w0 - lr * grad
print("SGD:  L2-in-loss vs. weight-decay step match:", np.allclose(w_l2_sgd, w_decay_sgd))
print("      max abs difference:", np.max(np.abs(w_l2_sgd - w_decay_sgd)))

# --- one Adam step: L2-in-gradient vs. decoupled (AdamW-style) weight decay ---
beta1, beta2, eps = 0.9, 0.999, 1e-8


def adam_step(g, w, lam, decoupled):
    grad_for_moments = g if decoupled else g + lam * w
    m = (1 - beta1) * grad_for_moments          # t=1: m_0=0
    v = (1 - beta2) * grad_for_moments ** 2      # t=1: v_0=0
    m_hat, v_hat = m / (1 - beta1), v / (1 - beta2)   # bias correction at t=1
    update = lr * m_hat / (np.sqrt(v_hat) + eps)
    if decoupled:
        update = update + lr * lam * w
    return w - update


w_l2_adam = adam_step(grad, w0, lam, decoupled=False)
w_decoupled_adam = adam_step(grad, w0, lam, decoupled=True)
print("\nAdam: L2-in-gradient vs. decoupled weight-decay step match:",
      np.allclose(w_l2_adam, w_decoupled_adam))
print("      max abs difference:", np.max(np.abs(w_l2_adam - w_decoupled_adam)))
assert not np.allclose(w_l2_adam, w_decoupled_adam), "Adam's two forms should genuinely differ"

SGD's two forms agree to floating-point precision, exactly as the
algebra predicts. Adam's two forms measurably disagree, because the
$\lambda w$ term entered the squared-gradient moving average $v$ in one
version and not the other, changing the per-parameter denominator that
every subsequent update divides by.

The rest of this notebook trains one small MLP — $[3072, 128, 64, 10]$,
He-initialised, ReLU hidden layers — on the 300-example training set from
"Setup" with **plain SGD**, so weight decay behaves exactly as derived
above with no adaptive-optimiser subtlety. That architecture is
deliberately larger than 300 examples calls for, so it overfits within a
modest number of epochs; every technique below is compared against this
one overfitting baseline.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def softmax(z):
    z = z - z.max(axis=0, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=0, keepdims=True)


def init_mlp(sizes, seed):
    rng = np.random.default_rng(seed)
    Ws, bs = [], []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        std = np.sqrt(2.0 / n_in)  # He initialisation (3a)
        Ws.append(rng.normal(0.0, std, size=(n_out, n_in)))
        bs.append(np.zeros((n_out, 1)))
    return Ws, bs


def forward(Ws, bs, X, dropout_prob=0.0, train=True, rng=None):
    """Generic forward pass with optional inverted dropout on every hidden
    layer. dropout_prob=0.0 reduces exactly to ordinary backprop -- one code
    path serves the baseline, weight-decay and augmentation experiments
    (dropout_prob=0.0) as well as the dropout experiment (dropout_prob>0.0)."""
    caches = {"a0": X, "masks": []}
    a = X
    L = len(Ws)
    for l in range(L - 1):
        z = Ws[l] @ a + bs[l]
        h = relu(z)
        if dropout_prob > 0.0 and train:
            keep = 1.0 - dropout_prob
            mask = (rng.random(h.shape) < keep).astype(np.float64) / keep
        else:
            mask = np.ones_like(h)
        a = h * mask
        caches["masks"].append(mask)
        caches[f"a{l + 1}"] = a
    z_out = Ws[-1] @ a + bs[-1]
    caches["probs"] = softmax(z_out)
    return caches["probs"], caches


def backward(Ws, caches, Y):
    L = len(Ws)
    B = Y.shape[1]
    grads_W, grads_b = [None] * L, [None] * L
    delta = (caches["probs"] - Y) / B
    for l in range(L - 1, -1, -1):
        a_prev = caches[f"a{l}"]
        grads_W[l] = delta @ a_prev.T
        grads_b[l] = delta.sum(axis=1, keepdims=True)
        if l > 0:
            da_prev = Ws[l].T @ delta
            mask_prev = caches["masks"][l - 1]
            delta = da_prev * mask_prev * (a_prev > 0)  # relu' via (a_prev>0); mask already carries 1/keep
    return grads_W, grads_b


def cross_entropy(probs, Y):
    return -np.mean(np.sum(Y * np.log(np.clip(probs, 1e-12, 1.0)), axis=0))


def accuracy(probs, labels):
    return (probs.argmax(axis=0) == labels).mean()


def random_crop_flip(images, rng, pad=4):
    """Pad each image by reflection, take a random 32x32 crop, and randomly
    flip it horizontally -- a minimal from-scratch version of the standard
    CIFAR-10 augmentation pipeline."""
    n, h, w, c = images.shape
    padded = np.pad(images, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="reflect")
    out = np.empty_like(images)
    for i in range(n):
        top = rng.integers(0, 2 * pad + 1)
        left = rng.integers(0, 2 * pad + 1)
        crop = padded[i, top:top + h, left:left + w, :]
        if rng.random() < 0.5:
            crop = crop[:, ::-1, :]
        out[i] = crop
    return out


def train_mlp_reg(weight_decay=0.0, dropout_prob=0.0, augment=False,
                   epochs=200, lr=0.02, batch_size=32, seed=SEED):
    rng = np.random.default_rng(seed)
    Ws, bs = init_mlp([3072, 128, 64, 10], seed)
    n = X_train.shape[1]
    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    for _ in range(epochs):
        X_epoch = random_crop_flip(images_train, rng).reshape(n, -1).T if augment else X_train
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            probs, caches = forward(Ws, bs, X_epoch[:, batch], dropout_prob=dropout_prob, train=True, rng=rng)
            grads_W, grads_b = backward(Ws, caches, Y_train[:, batch])
            for l in range(len(Ws)):
                Ws[l] -= lr * (grads_W[l] + weight_decay * Ws[l])
                bs[l] -= lr * grads_b[l]
        train_probs, _ = forward(Ws, bs, X_train, train=False)
        test_probs, _ = forward(Ws, bs, X_test, train=False)
        history["train_loss"].append(cross_entropy(train_probs, Y_train))
        history["test_loss"].append(cross_entropy(test_probs, Y_test))
        history["train_acc"].append(accuracy(train_probs, labels_train))
        history["test_acc"].append(accuracy(test_probs, labels_test))
    return Ws, bs, history


_, _, baseline_history = train_mlp_reg(weight_decay=0.0)
_, _, wd_history = train_mlp_reg(weight_decay=0.02)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for name, h in [("baseline (no weight decay)", baseline_history), ("weight decay = 0.02", wd_history)]:
    axes[0].plot(h["train_loss"], "--", alpha=0.6)
    axes[0].plot(h["test_loss"], label=name)
    axes[1].plot(h["train_acc"], "--", alpha=0.6)
    axes[1].plot(h["test_acc"], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (solid=test, dashed=train)")
axes[0].set_title("Loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy (solid=test, dashed=train)")
axes[1].set_title("Accuracy"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

baseline_gap = baseline_history["train_acc"][-1] - baseline_history["test_acc"][-1]
wd_gap = wd_history["train_acc"][-1] - wd_history["test_acc"][-1]
print(f"final train-test accuracy gap -- baseline: {baseline_gap:.3f}, weight decay: {wd_gap:.3f}")
assert wd_gap < baseline_gap, "weight decay should narrow the generalisation gap"

The baseline reaches 100% train accuracy while test accuracy plateaus far
below — a wide, clean generalisation gap, exactly the overfitting regime
this notebook is built to demonstrate on a 300-example training set with a
`[3072,128,64,10]` network. Weight decay narrows that gap by holding train
accuracy back: constraining the weights away from the large values that
would let the network fit this particular sample exactly costs some
training accuracy, and at this small scale that cost is not fully repaid
in test accuracy either — a realistic outcome for a single, small-sample
run, and exactly why weight decay's benefit is stated here as "smaller
gap", not "higher test accuracy": the mechanism (constrain the weights) is
what 3a and the derivation above guarantee, not a specific numeric
payoff on every dataset.

## Dropout

Dropout (Srivastava et al., 2014) regularises a different way: instead of
constraining weight magnitude, it randomly deletes units during training.
For a hidden layer with pre-dropout activation $h_j$ and independent
Bernoulli mask $m_j\sim\text{Bernoulli}(q)$ with keep probability $q$, the
layer's output during training is

$$a_j = \frac{m_j}{q}\,h_j \qquad(\text{"inverted" dropout}).$$

Every one of the $2^n$ possible mask patterns across $n$ units defines a
different, smaller sub-network, and because every sub-network shares the
same underlying weights, training with dropout approximates training an
**exponentially large ensemble of weight-sharing sub-networks**
simultaneously — one sampled fresh at every forward pass. An ensemble
generalises better than any single member because the members' individual
errors partially cancel on average; dropout buys (an approximation to)
that benefit without literally training or storing more than one network.

**The train/test scaling rule.** Ensembling means averaging the *predictions*
of many sub-networks, which is expensive to do literally. Instead, the
standard approximation is to run the *entire* network, no units dropped,
at test time. For this single full-network pass to approximate the
ensemble average rather than silently changing the activation scale
(every unit now always contributes, instead of only a $q$ fraction on
average), the training-time scaling by $1/q$ above is exactly what makes
$\mathbb E_{m}[a_j] = h_j$ — matching what the always-on test-time network
computes with no further adjustment needed. (The equivalent, older
formulation scales weights down by $q$ at *test* time instead and uses no
scaling during training; inverted dropout, used here and in
`nn.Dropout`, moves that same factor to train time so evaluation needs no
special case.)

In [ ]:
_, _, dropout_history = train_mlp_reg(dropout_prob=0.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for name, h in [("baseline (no dropout)", baseline_history), ("dropout p=0.5", dropout_history)]:
    axes[0].plot(h["train_loss"], "--", alpha=0.6)
    axes[0].plot(h["test_loss"], label=name)
    axes[1].plot(h["train_acc"], "--", alpha=0.6)
    axes[1].plot(h["test_acc"], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (solid=test, dashed=train)")
axes[0].set_title("Loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy (solid=test, dashed=train)")
axes[1].set_title("Accuracy"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

dropout_gap = dropout_history["train_acc"][-1] - dropout_history["test_acc"][-1]
print(f"final train-test accuracy gap -- baseline: {baseline_gap:.3f}, dropout: {dropout_gap:.3f}")
assert dropout_gap < baseline_gap, "dropout should narrow the generalisation gap"

Dropout's training curve is visibly noisier — every batch trains a
different sampled sub-network, so the loss fluctuates more than the clean
baseline — and, as with weight decay, training accuracy no longer runs
away to 100% as fast. The narrower final gap is the same story from a
different mechanism: rather than shrinking the weights directly, dropout
prevents units from co-adapting to compensate for each other's specific
quirks on this particular training sample, which is precisely what an
ensemble of many differently-damaged sub-networks would also prevent.

## Early Stopping

The baseline curve in "Weight Decay and L2" already contains everything
early stopping needs: test loss falls together with training loss at
first, then — once the network has extracted the genuine signal and starts
fitting sample-specific noise instead — test loss stops improving or rises
while training loss keeps falling. **Early stopping** simply means
monitoring that test (or held-out validation) loss during training and
keeping the weights from whichever epoch minimised it, rather than the
weights from the final epoch. It costs nothing beyond training the network
once, since the checkpoint of interest is found by looking back at a
curve that was going to be computed anyway.

In [ ]:
best_epoch = int(np.argmin(baseline_history["test_loss"]))
final_epoch = len(baseline_history["test_loss"]) - 1

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(baseline_history["train_loss"], "--", label="train loss")
ax.plot(baseline_history["test_loss"], label="test loss")
ax.axvline(best_epoch, color="C2", linestyle=":", label=f"early-stopping point (epoch {best_epoch})")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Early stopping on the unregularised baseline")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

print(f"test accuracy at early-stopping epoch {best_epoch}: {baseline_history['test_acc'][best_epoch]:.3f}")
print(f"test accuracy at final epoch {final_epoch}:         {baseline_history['test_acc'][final_epoch]:.3f}")

Stopping at the epoch of minimum test loss reaches test accuracy at
least as good as training to the bitter end, without touching the weight
update rule or the data pipeline at all — the cheapest regulariser in this
notebook, precisely because it only requires *watching* training that was
already happening rather than changing it.

## Data Augmentation

Every technique so far regularised the optimiser or the architecture.
**Data augmentation** regularises the other input to the problem: the
data itself. `random_crop_flip`, defined in "Weight Decay and L2" above,
generates a fresh, label-preserving transformation of every training image
on every epoch — a random 4-pixel-padded crop back to $32\times32$, and a
random horizontal flip. The network never sees exactly the same pixels
twice, so memorising this specific 300-image sample stops being a
reliable way to reduce training loss; the only features worth learning are
ones robust to the small translations and reflections a real photograph
of the same object could equally have had. Augmentation effectively trains
against a much larger, implicitly defined dataset than the 300 images
actually stored.

In [ ]:
_, _, augment_history = train_mlp_reg(augment=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for name, h in [("baseline (no augmentation)", baseline_history), ("random crop + flip", augment_history)]:
    axes[0].plot(h["train_loss"], "--", alpha=0.6)
    axes[0].plot(h["test_loss"], label=name)
    axes[1].plot(h["train_acc"], "--", alpha=0.6)
    axes[1].plot(h["test_acc"], label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (solid=test, dashed=train)")
axes[0].set_title("Loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy (solid=test, dashed=train)")
axes[1].set_title("Accuracy"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

augment_gap = augment_history["train_acc"][-1] - augment_history["test_acc"][-1]
print(f"final train-test accuracy gap -- baseline: {baseline_gap:.3f}, augmentation: {augment_gap:.3f}")
assert augment_gap < baseline_gap, "augmentation should narrow the generalisation gap"

Training accuracy under augmentation climbs more slowly — the network is
chasing a moving target, a freshly transformed batch every epoch, rather
than 300 fixed images it can eventually memorise outright — and the
train/test gap closes further than the unregularised baseline, the same
qualitative effect as weight decay and dropout achieved through entirely
different mechanisms: all three make the training signal harder for the
network to satisfy by memorisation alone.

## Double Descent

"Bias and Variance in Deep Networks" stopped the random-features sweep
just short of $p=n=50$, exactly where the classical U-shape predicts test
error should be at its worst — and it was. What the classical argument
does not anticipate is what happens on the *other* side: once the model
has strictly more capacity than training examples ($p>n$), `fit_min_norm`
still returns a well-defined answer (the minimum-norm solution among the
infinitely many that fit the training data exactly), and its test error
does not keep getting worse.

In [ ]:
for p in [60, 80, 120, 200, 350, 600, 1000, 1600]:
    dd_results[p] = double_descent_point(p, seed=SEED)

ps_full = sorted(dd_results)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ps_full, [dd_results[p][0] for p in ps_full], marker="o", markersize=4, label="train error")
ax.plot(ps_full, [dd_results[p][1] for p in ps_full], marker="o", markersize=4, label="test error")
ax.axvline(N_DD_TRAIN, color="gray", linestyle=":", label=f"n = {N_DD_TRAIN} (interpolation threshold)")
ax.set_xscale("log")
ax.set_xlabel("number of random features (model capacity, log scale)")
ax.set_ylabel("classification error")
ax.set_title("Double descent: the full sweep")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

under = min(p for p in ps_full if p < N_DD_TRAIN)
peak = max(dd_results, key=lambda p: dd_results[p][1])
over = max(ps_full)
print(f"test error well under capacity (p={under}):  {dd_results[under][1]:.3f}")
print(f"test error at the worst point (p={peak}):     {dd_results[peak][1]:.3f}")
print(f"test error well over capacity (p={over}):  {dd_results[over][1]:.3f}")
assert dd_results[over][1] < dd_results[peak][1], "test error should come back down past the interpolation threshold"

Test error peaks almost exactly at $p=n$ — the interpolation
threshold, where the model has just enough capacity to fit the training
set exactly and no slack left to do so robustly — then **descends a second
time** as capacity keeps growing well past it. This is "double descent":
the classical U-shape is not wrong, it is the *first half* of a longer
curve. Past the interpolation threshold, `fit_min_norm`'s minimum-norm
solution increasingly resembles a smooth interpolant rather than a
noise-fitting one — with many more features than constraints, the
optimisation problem has enormous freedom, and *minimum norm among all
exact fits* turns out empirically to favour simple, well-generalising
solutions over baroque ones. This is also why heavily overparameterised
deep networks — sitting far to the right of this plot in parameter count —
routinely reach zero training error and still generalise well: they are
not violating the bias-variance tradeoff, they are operating in the region
of it the classical derivation never considered.

## Key Takeaways

- The classical **bias-variance decomposition**,
  $\mathbb E[(y-\hat f)^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$,
  correctly predicts a U-shaped test-error curve as capacity approaches the
  number of training examples — but implicitly assumes variance keeps
  growing indefinitely past that point, which is exactly where modern
  overparameterised networks operate and the assumption fails.
- **Weight decay and $L_2$ regularisation are the same update for plain
  SGD** ($w\leftarrow(1-\eta\lambda)w-\eta\nabla_wJ$, verified to
  floating-point precision), but diverge for Adam because the $L_2$
  gradient term, if folded into the loss, gets rescaled by the adaptive
  per-parameter denominator — the precise reason AdamW keeps weight decay
  decoupled from the moment estimates.
- **Dropout** trains an implicit ensemble of $2^n$ weight-sharing
  sub-networks by randomly zeroing units each forward pass; the
  **inverted-dropout scaling rule** ($a_j = m_j h_j/q$ at train time, full
  network at test time) is exactly what makes running the complete
  network once at test time approximate that ensemble's average.
- **Early stopping** costs nothing beyond training once: the test-loss
  curve already reveals the epoch to stop at, without changing the update
  rule or the data at all.
- **Data augmentation** (random crop and flip, from scratch) regularises
  the data itself rather than the optimiser: a network trained on a
  freshly transformed batch every epoch cannot rely on memorising the
  original pixels, which is what narrows its generalisation gap.
- **Double descent** reproduced on real CIFAR-10 data: test error peaks at
  the interpolation threshold ($p=n$) exactly as the classical picture
  predicts, then descends a second time as capacity grows further past
  it — the empirical result that motivates treating "more capacity than
  data" as a regime to understand, not a mistake to avoid.